# Dynamic document-slice radius prediction (k ∈ {0, 1, 2, 3})

Extension of *Efficient Local RAG via Neighboring Document Slice Contextualization*: instead of a fixed slice radius
(k = 3 in the paper), predict **per chunk** how many neighbouring chunks the contextualiser needs, with the **same**
local model that later writes the context (`gemma3:4b-it-qat`), so nothing extra has to live in the 8 GB of VRAM.

**Design (two tiers, both Green-AI cheap)**

| Tier | What | Cost | Output |
|---|---|---|---|
| 1 | Deterministic **table gate** on the Docling text (host CPU, regex) | ~0 ms | `k = 3` for any chunk that carries a table (caption and/or serialised cells) |
| 2 | **Single forward pass** of the SLM, `num_predict = 1`, argmax over four answer tokens read from the log-probs | ~160 ms on the RTX 4060 Laptop (8 GB) | `k ∈ {0,1,2,3}` |

**Semantics of k** (the slice for chunk *i* is chunks *i−k … i+k*, boundary-extended as in the paper)

| k | Meaning | Typical content |
|---|---|---|
| 0 | chunk is already a summary or self-evident boilerplate → **skip contextualisation** | abstract, conclusion, reference list, acknowledgements, bios, metadata |
| 1 | light context | introduction, background/related work, dataset / hardware / setup descriptions |
| 2 | **default** ("if unsure, 2") | formulas with *where …*, results/discussion pointing to figures, tables, scenarios, "the proposed method", mid-sentence cuts, section stubs |
| 3 | widest window | tables (Tier 1) and near-meaningless fragments (bullet items, pseudo-code, bare lists) |

**Headline numbers** (gold set `dynamic_slice_length/slice_radius_gold.json`, 119 verbatim Docling chunks from 24 of the 25 papers)

| | Accuracy |
|---|---|
| Original notebook (9 SOFM chunks, digits 0–2 prompt) | 4 / 9 |
| This version on the same 9 chunks | **8 / 9** |
| Table gate (20 tables + 99 non-tables, incl. 4 formula-heavy negatives) | 119 / 119, 0 false positives; 42 / 382 corpus chunks gated |
| SLM exact match (99 non-table chunks) | **65.7 %** (within ±1 step: 86.9 %, MAE 0.47) |
| SLM exact match after leave-one-document-out calibration | 68.7 % |
| Overall exact match (gate + SLM) | 71.4 % |
| Corpus-wide (382 chunks, 60 s): slice text handed to the contextualiser vs. fixed k = 3 | **53 %**, with 109 chunks (k = 0) needing no contextualisation call at all |

Inputs are the pre-extracted, pre-chunked JSONs in `split_documents/` (Docling is non-deterministic, so PDFs are never
re-processed here). Requires a running Ollama (`ollama serve`) with `gemma3:4b-it-qat` pulled.

In [1]:
import glob, json, math, re, time
from collections import Counter, defaultdict
from pathlib import Path

import numpy as np
import ollama

SPLIT_DIR = Path("split_documents")                 # Docling HybridChunker output, one JSON list per paper
GOLD_PATH = Path("dynamic_slice_length/slice_radius_gold.json")
PRED_OUT = Path("dynamic_slice_length/predicted_k_all_chunks.json")

OLLAMA_MODEL_NAME = "gemma3:4b-it-qat"              # same model as the contextualiser -> no extra VRAM
K_MAX, K_DEFAULT = 3, 2                             # K_DEFAULT is the fallback when nothing can be parsed
QUERY_HEAD, QUERY_TAIL = 1500, 500                  # chars of the chunk shown to the SLM (head [...] tail)

def load_doc(pattern):
    return json.load(open(glob.glob(str(SPLIT_DIR / pattern))[0]))

CORPUS = {Path(p).name[:-5]: json.load(open(p)) for p in sorted(glob.glob(str(SPLIT_DIR / "*.json")))}
print(f"{len(CORPUS)} documents, {sum(map(len, CORPUS.values()))} chunks")

25 documents, 382 chunks


## Tier 1 — deterministic table gate

Docling's `HybridChunker` linearises a table into **one line** of cells of the form `<row label>, <column header> = <value>.`
(e.g. `A, Vergence [8].mm = 33.678. A, Vergence [8].deg = 0.248. …`) and emits the caption as its **own line**
(`TABLE 1. Title`, `TABLE I` + all-caps title on the next line, `Table 1 | Title`, `Table 1: Title`, `Table 1 Profits …`).
Markdown pipes are never produced, so the old `|` / `:---` heuristics are useless here.

The gate fires on either signature:

1. **≥ 3 serialised cells on a single line** with a `=` density of at most 100 chars per `=`. The cell regex requires a
   non-space character right before the terminating period, which is what separates real cells from maths prose such as
   `V = { 1, . . . , m }` (ellipsis dots) or `gx = 2.0087(3), gy = …` (no terminator).
2. **A caption at the start of a line.** In-text mentions (`… shown in Table 1.`, `Table 1 shows …`) are mid-line or
   followed by a lowercase word and therefore do not match. Nature-style `Table 1 |` is accepted anywhere because Docling
   sometimes glues it to the first cell.

A chunk that only carries the caption (Docling dropped the body) is still gated to k = 3: the table body is then in a
neighbouring chunk and the caption alone is decontextualised.

In [2]:
_CELL_RE = re.compile(r", [^,=\n]{1,80} = [^=\n]{0,80}\S\.(?: |$)")
_CAPTION_RE = re.compile(
    r"(?m)^[ \t]*(?:TABLE|Table)[ \t]+(?:\d{1,2}|[IVXLC]{1,5})\b"
    r"(?:[ \t]*[.:|][ \t]*\S"      # 'TABLE 1. Title' | 'Table 1: Title' | 'Table 1 | Title'
    r"|[ \t]+[A-Z][A-Za-z]"        # 'TABLE I SAMPLE BINS ...' | 'Table 1 Profits and ...'
    r"|[ \t]*$)"                   # 'TABLE I' alone, all-caps title on the next line (IEEE Trans.)
)
_PIPE_CAPTION_RE = re.compile(r"\bTable[ \t]+\d{1,2}[ \t]*\|")


def table_evidence(text: str):
    """Return a short reason if the chunk contains a Docling-serialised table, else None."""
    for line in text.split("\n"):
        cells = _CELL_RE.findall(line)
        if len(cells) >= 3 and len(line) / max(1, line.count(" = ")) <= 100:
            return f"{len(cells)} serialised cells on one line: {line[:60]!r}"
    m = _CAPTION_RE.search(text)
    if m:
        return f"caption: {text[m.start():m.end() + 40].splitlines()[0]!r}"
    m = _PIPE_CAPTION_RE.search(text)
    if m:
        return f"pipe caption: {text[m.start():m.start() + 40]!r}"
    return None


def is_structural_table(text: str) -> bool:
    return table_evidence(text) is not None

In [3]:
# Sweep the whole corpus: which chunks does the gate fire on, and does it stay quiet on the known hard negatives?
flagged = [(doc, i, table_evidence(c["text"])) for doc, chunks in CORPUS.items() for i, c in enumerate(chunks) if is_structural_table(c["text"])]
print(f"gated chunks: {len(flagged)} / {sum(map(len, CORPUS.values()))}\n")
for doc, i, ev in flagged:
    print(f"  {doc[:40]:40s} #{i:2d}  {ev[:85]}")

# formula-heavy prose / in-text 'Table 1' mentions that must NOT fire
HARD_NEGATIVES = [("A_Resource*", 9), ("Scalable*", 12), ("Electron*", 2), ("s41598-021*", 2), ("s41598-020*", 1), ("Thermal*", 14), ("A_Hybrid_Gaze*", 11)]
for pat, i in HARD_NEGATIVES:
    assert not is_structural_table(load_doc(pat)[i]["text"]), (pat, i)
print("\nall hard negatives pass")

gated chunks: 42 / 382

  A_Conceptual_Framework_and_Recommendatio #14  caption: 'Table 1: Data and artifacts made open from the Gan'
  A_Feature_Fusion_Based_Indicator_for_Tra #14  caption: 'TABLE 1. Performance of several training-free indi'
  A_Feature_Fusion_Based_Indicator_for_Tra #15  caption: 'TABLE 2. Performance of several training-free indi'
  A_Feature_Fusion_Based_Indicator_for_Tra #16  caption: 'TABLE 3. Performance of FI 4 on NAS-Bench-201, mea'
  A_Feature_Fusion_Based_Indicator_for_Tra #19  caption: 'TABLE 4. KROCC of indicators with different weight'
  A_Feature_Fusion_Based_Indicator_for_Tra #20  caption: 'TABLE 5. Comparison with different combination of '
  A_Feature_Fusion_Based_Indicator_for_Tra #21  caption: 'TABLE 6. The results from Shapiro-Wilk tests on CI'
  A_Hybrid_Gaze_Distance_Estimation_via_Cr #10  36 serialised cells on one line: 'A, Section/Scenario. = S1. A, Vergence [8].mm = 33.
  A_Resource_Allocation_Model_Based_on_Tru #10  caption: 'TABLE 1. Descr

## Tier 2 — single-forward-pass SLM classifier

What worked, and what did not (details in the ablation table further down):

* **Ask a genre question, not a "how many neighbours" question.** A 4B model cannot reason about what a downstream
  summariser will need, but it can tell an abstract from a results paragraph. The four genres map 1:1 onto k
  (A → 0, B → 1, C → 2, D → 3) and the mapping is done in code.
* **Answer with letters, read the log-probs.** With `num_predict = 1` Ollama returns `top_logprobs`; the decision is the
  argmax over the four letter tokens only, so a stray `**` or newline can never derail parsing. If log-probs are missing
  the emitted token is parsed, and if that fails the fallback is `K_DEFAULT = 2`.
* **Ten few-shot exemplars from held-out material** (the Gaze paper, one chunk of the Nature machine-behaviour paper,
  one author-contribution line), clipped to ≤ 900 chars. Adding more exemplars, showing their tails, or appending a line of
  cheap surface signals (word count, equation placeholders, position in the document, …) all made the model *worse*.
* **Clip the query** to the first 1500 and last 500 characters: enough to see the opener and a mid-sentence cut, and the
  whole prompt stays around 2.3k tokens → ~160 ms per chunk on the 4060 (median 150 ms).
* Keep `temperature = 0`; the run is deterministic, which matters for the paper's reproducibility claims.

In [4]:
LETTER_TO_K = {"A": 0, "B": 1, "C": 2, "D": 3}
K_TO_LETTER = {v: k for k, v in LETTER_TO_K.items()}

SYSTEM_PROMPT = (
    "Classify a passage taken from a scientific paper into one of four genres. Answer with a single letter.\n\n"
    "A - Whole-paper summary or boilerplate. The abstract; a conclusion or summary of the whole paper (these typically open "
    "with 'In this paper', 'In conclusion', 'In summary', 'This paper has proposed', 'This article has proposed', 'We have "
    "proposed'); or non-content matter: reference list, acknowledgements, funding statement, author biographies or "
    "contributions, competing-interests statement, publication metadata, licence text, layout garbage. A conclusion is A "
    "even when it is full of numbers, acronyms and technical terms.\n"
    "B - Self-contained exposition. Introduction, motivation, background or related work, or a description of a dataset, "
    "device, system, software or experimental setup that names its subject in full and is understandable without the rest "
    "of the paper. Contains no equation placeholders. Introductions and related-work paragraphs are B even when technical; "
    "they typically cite many references such as [3], [7]-[9] and describe what other authors did.\n"
    "C - Section-internal technical body. Methods, derivations, results or discussion that continue an argument of their "
    "section: '<!-- formula-not-decoded -->' placeholders with 'where ...' definitions, references to figures, tables, "
    "scenarios or equations, 'the proposed method', undefined acronyms or symbols, a one- or two-sentence section stub "
    "('This section presents ...'), or text that starts or stops mid-sentence.\n"
    "D - Fragment. A couple of bullet items, pseudo-code, a bare list of inputs, a dangling caption, or a stub that only "
    "names undefined acronyms; meaningless without its surroundings.\n\n"
    "Reply with the letter only."
)


def clip(text, head=QUERY_HEAD, tail=QUERY_TAIL):
    text = text.strip()
    if len(text) <= head + tail + 20:
        return text
    return text[:head].rstrip() + (" [...] " + text[-tail:].lstrip() if tail else " [...]")


def build_few_shots():
    """Exemplars come from material that is held out of the gold set (verbatim Docling chunks, head-clipped)."""
    gaze = load_doc("A_Hybrid_Gaze*")        # whole paper held out
    nat = load_doc("s41586*")                # only chunk 12 used here
    srep = load_doc("s41598-020*")           # only chunk 10 used here
    refs = "\n".join(gaze[13]["text"].split("\n")[:3])
    return [
        (clip(gaze[1]["text"], 900, 0), 0),    # affiliations + ABSTRACT
        (clip(gaze[8]["text"], 700, 0), 2),    # confidence-measure formulas with 'where'
        (clip(gaze[3]["text"], 650, 0), 1),    # related work, methods named
        (nat[12]["text"], 3),                  # two bullet questions from a figure
        (refs, 0),                             # reference list excerpt
        (clip(gaze[11]["text"], 600, 0), 2),   # results pointing to Fig. 7, Eq. 1/3, segments
        (clip(gaze[5]["text"], 650, 0), 1),    # method overview, devices named
        (clip(gaze[12]["text"], 700, 0), 0),   # conclusion 'This paper proposed ...'
        (clip(gaze[9]["text"], 600, 0), 1),    # experimental setup
        (srep[10]["text"], 0),                 # author contributions (87 chars)
    ]


FEW_SHOTS = build_few_shots()


def build_messages(chunk_text):
    parts = [f"Passage:\n{shown}\nGenre: {K_TO_LETTER[k]}\n" for shown, k in FEW_SHOTS]
    parts.append(f"Passage:\n{clip(chunk_text)}\nGenre:")
    return [{"role": "system", "content": SYSTEM_PROMPT}, {"role": "user", "content": "\n".join(parts)}]


def slm_scores(chunk_text, model_name=OLLAMA_MODEL_NAME):
    """One forward pass, num_predict=1. Returns ({k: logprob} over the letter tokens, raw response)."""
    resp = ollama.chat(
        model=model_name, messages=build_messages(chunk_text),
        options={"temperature": 0.0, "num_predict": 1, "top_p": 1.0},
        logprobs=True, top_logprobs=20,
    )
    scores = {}
    if resp.logprobs:
        for tl in resp.logprobs[0].top_logprobs:
            tok = tl.token.strip().upper()
            if tok in LETTER_TO_K and LETTER_TO_K[tok] not in scores:
                scores[LETTER_TO_K[tok]] = tl.logprob
    if not scores:                                             # no log-probs -> parse the emitted token -> default
        m = re.search(r"[A-Da-d]", resp.message.content or "")
        scores = {LETTER_TO_K[m.group(0).upper()] if m else K_DEFAULT: 0.0}
    return scores, resp


def decide(scores, bias=None):
    """Argmax over letter log-probs, optionally shifted by an additive calibration bias vector."""
    return max(scores, key=lambda k: scores[k] + (bias[k] if bias is not None else 0.0))


def to_probs(scores):
    z = np.array([scores.get(k, -30.0) for k in range(K_MAX + 1)])
    p = np.exp(z - z.max()); return p / p.sum()


def predict_dynamic_k(chunk_text, bias=None):
    """Tier 1 gate, then Tier 2 SLM. Returns (k, tier)."""
    if is_structural_table(chunk_text):
        return 3, "table"
    scores, _ = slm_scores(chunk_text)
    return decide(scores, bias), "slm"


# warm-up (loads the model into VRAM) + show what the prompt looks like
t0 = time.perf_counter(); _s, _r = slm_scores(FEW_SHOTS[0][0]); print(f"warm-up {time.perf_counter()-t0:.1f}s, prompt tokens {_r.prompt_eval_count}, probs {np.round(to_probs(_s), 3)}")
print("\n--- system prompt ---\n" + SYSTEM_PROMPT[:600] + " ...")
print(f"\n--- few-shots: {len(FEW_SHOTS)} exemplars, labels {[K_TO_LETTER[k] for _, k in FEW_SHOTS]} ---")

warm-up 0.2s, prompt tokens 1972, probs [0.992 0.005 0.003 0.001]

--- system prompt ---
Classify a passage taken from a scientific paper into one of four genres. Answer with a single letter.

A - Whole-paper summary or boilerplate. The abstract; a conclusion or summary of the whole paper (these typically open with 'In this paper', 'In conclusion', 'In summary', 'This paper has proposed', 'This article has proposed', 'We have proposed'); or non-content matter: reference list, acknowledgements, funding statement, author biographies or contributions, competing-interests statement, publication metadata, licence text, layout garbage. A conclusion is A even when it is full of numbers,  ...

--- few-shots: 10 exemplars, labels ['A', 'C', 'B', 'D', 'A', 'C', 'B', 'A', 'B', 'A'] ---


## Gold set

`dynamic_slice_length/slice_radius_gold.json` holds 119 hand-labelled chunks (30 × k=0, 25 × k=1, 37 × k=2, 27 × k=3 of
which 20 are tables). Every `text` is the verbatim Docling chunk, keyed by `document` + `chunk_index` + `id`, so the set can
be regenerated or extended without touching the PDFs. Each item carries a one-line `rationale` and a `tier`:

* `"table"` — must be caught by the Tier-1 gate;
* `"slm"` — must **not** be gated (four of these are formula-heavy hard negatives) and is scored by the SLM.

The nine chunks of the original notebook (SOFM paper) are all included. Labelling rubric: 0 = already a summary or
boilerplate; 1 = self-contained exposition; 2 = continues an argument / formulas / results / mid-sentence cut / stub
(default when unsure, following the paper's ablation); 3 = table or near-meaningless fragment.

In [5]:
GOLD = json.load(open(GOLD_PATH))["items"]
ORIGINAL_NINE = {"SOFM #1", "SOFM #14", "SOFM #4", "SOFM #8", "SOFM #10", "SOFM #6", "SOFM #7", "SOFM #12", "SOFM #11"}

rows = []
for it in GOLD:
    t0 = time.perf_counter()
    ev = table_evidence(it["text"])
    if ev:
        scores, tier, ptoks = {3: 0.0}, "table", 0
    else:
        scores, resp = slm_scores(it["text"]); tier, ptoks = "slm", resp.prompt_eval_count
    rows.append({**it, "scores": scores, "pred": decide(scores), "pred_tier": tier, "ms": (time.perf_counter() - t0) * 1000,
                 "prompt_tokens": ptoks, "evidence": ev})

print(f"{'':2s}{'item':20s} {'gold':>4s} {'pred':>4s} {'tier':6s} {'ms':>6s}  p(k=0..3) / gate evidence")
for r in rows:
    flag = "  " if r["pred"] == r["expected_k"] else "XX"
    info = " ".join(f"{p:.2f}" for p in to_probs(r["scores"])) if r["pred_tier"] == "slm" else r["evidence"][:60]
    print(f"{flag}{r['label']:20s} {r['expected_k']:4d} {r['pred']:4d} {r['pred_tier']:6s} {r['ms']:6.0f}  {info}")


def report(rows, key, title):
    conf = defaultdict(Counter)
    for r in rows: conf[r["expected_k"]][r[key]] += 1
    slm = [r for r in rows if r["tier"] == "slm"]
    ok = sum(r[key] == r["expected_k"] for r in rows); sok = sum(r[key] == r["expected_k"] for r in slm)
    w1 = sum(abs(r[key] - r["expected_k"]) <= 1 for r in slm); mae = sum(abs(r[key] - r["expected_k"]) for r in slm) / len(slm)
    print(f"\n== {title} ==\nconfusion (rows = gold k, cols = predicted k 0..3):")
    for g in range(4): print(f"  gold {g}: " + " ".join(f"{conf[g][p]:3d}" for p in range(4)))
    print(f"overall exact {ok}/{len(rows)} = {ok/len(rows):.1%} | SLM exact {sok}/{len(slm)} = {sok/len(slm):.1%} | "
          f"SLM within ±1 {w1/len(slm):.1%} | SLM MAE {mae:.2f}")

gate_ok = sum((r["pred_tier"] == "table") == (r["tier"] == "table") for r in rows)
print(f"\ntable gate: {gate_ok}/{len(rows)} correct, false positives: {[r['label'] for r in rows if r['pred_tier']=='table' and r['tier']!='table']}")
report(rows, "pred", "raw letter argmax")
nine = [r for r in rows if r["label"] in ORIGINAL_NINE]
print(f"original 9 SOFM test chunks: {sum(r['pred']==r['expected_k'] for r in nine)}/9 (was 4/9)")
lat = [r["ms"] for r in rows if r["pred_tier"] == "slm"]; ptk = [r["prompt_tokens"] for r in rows if r["pred_tier"] == "slm"]
print(f"SLM latency: mean {np.mean(lat):.0f} ms, median {np.median(lat):.0f} ms, max {max(lat):.0f} ms | prompt tokens mean {np.mean(ptk):.0f}, max {max(ptk)}")

  item                 gold pred tier       ms  p(k=0..3) / gate evidence
  SOFM #1                 0    0 slm       227  1.00 0.00 0.00 0.00
  SOFM #14                0    0 slm       152  1.00 0.00 0.00 0.00
XXFeature #1              0    2 slm       219  0.01 0.04 0.95 0.00
XXFeature #22             0    2 slm       156  0.11 0.10 0.78 0.01
  Feature #0              0    0 slm       100  0.99 0.00 0.00 0.00
XXFeature #23             0    2 slm       272  0.18 0.09 0.72 0.01
XXElectron #0             0    2 slm       167  0.00 0.00 0.99 0.00
XXElectron #8             0    2 slm       152  0.00 0.00 1.00 0.00
XXConceptual #1           0    1 slm       161  0.00 0.97 0.02 0.00
  Conceptual #17          0    0 slm       109  0.99 0.00 0.00 0.00
  Conceptual #19          0    0 slm       287  0.99 0.00 0.00 0.00
XXRubidium #7             0    2 slm       207  0.00 0.00 1.00 0.00
  Rubidium #10            0    0 slm       205  0.99 0.00 0.00 0.00
  Stock #15               0    0 slm      

## Optional: contextual calibration of the letter log-probs

The raw letter distribution is skewed towards C (k = 2). *Contextual calibration* (Zhao et al., 2021) fits an additive bias
per class on the gold log-probs by minimising cross-entropy, and the deployed decision becomes `argmax(logp + bias)` —
still one forward pass. The honest estimate below is **leave-one-document-out**: the bias is fitted on the other papers and
applied to the held-out paper.

The gain is small (≈ +3 points) and the fitted bias pushes *away* from k = 2, i.e. against the paper's prior ("if unsure,
k = 2"), so calibration is **off by default** (`USE_CALIBRATION = False`). Flip it on if the gold set grows and the CV gain
becomes convincing.

In [6]:
def fit_bias(logp_rows, gold, l2=0.05, steps=400, lr=0.5):
    L, y, b = np.asarray(logp_rows, float), np.asarray(gold), np.zeros(K_MAX + 1)
    onehot = np.eye(K_MAX + 1)[y]
    for _ in range(steps):
        z = L + b; z -= z.max(1, keepdims=True); p = np.exp(z); p /= p.sum(1, keepdims=True)
        grad = (p - onehot).mean(0) + l2 * b; grad[0] = 0.0          # anchor class 0
        b -= lr * grad
    return b

def logp_row(scores): return [scores.get(k, -30.0) for k in range(K_MAX + 1)]

slm_rows = [r for r in rows if r["pred_tier"] == "slm"]
for d in sorted({r["document"] for r in slm_rows}):                   # leave-one-document-out
    train = [r for r in slm_rows if r["document"] != d]
    b = fit_bias([logp_row(r["scores"]) for r in train], [r["expected_k"] for r in train])
    for r in slm_rows:
        if r["document"] == d: r["pred_cal"] = decide(r["scores"], b)
for r in rows: r.setdefault("pred_cal", r["pred"])
report(rows, "pred_cal", "calibrated, leave-one-document-out")

CALIBRATION_BIAS = fit_bias([logp_row(r["scores"]) for r in slm_rows], [r["expected_k"] for r in slm_rows])
USE_CALIBRATION = False
print("\nbias fitted on all gold (deployment candidate):", np.round(CALIBRATION_BIAS, 3).tolist(), "| USE_CALIBRATION =", USE_CALIBRATION)


== calibrated, leave-one-document-out ==
confusion (rows = gold k, cols = predicted k 0..3):
  gold 0:  16   5   9   0
  gold 1:   0  13  12   0
  gold 2:   0   3  34   0
  gold 3:   0   1   1  25
overall exact 88/119 = 73.9% | SLM exact 68/99 = 68.7% | SLM within ±1 89.9% | SLM MAE 0.41

bias fitted on all gold (deployment candidate): [0.0, 0.308, -2.802, 0.023] | USE_CALIBRATION = False


## Ablation log (same gold set, temperature 0, `gemma3:4b-it-qat`)

SLM exact match on the 99 non-table gold chunks; "cal." = leave-one-document-out calibrated.

| # | Prompt / answer format | Extras | Exemplars | raw | cal. |
|---|---|---|---|---|---|
| 1 | digits 0–3, "how many neighbours does a summariser need", *"if torn choose 2"* | – | 8 | 42.4 % | – |
| 2 | digits 0–3, per-level cue lists, no tie rule | – | 10 | 49.5 % | 54.5 % |
| 3 | as 2 | + surface-signals line (words, `#formula`, fig/table refs, mid-sentence flags, openers) | 10 | 50.5 % | 53.5 % |
| 4 | **letters A–D, genre framing** | – | 10 | 63.6 % | 67.7 % |
| 5 | as 4 | + signals line + position in document | 10 | 60.6 % | 63.6 % |
| 6 | as 4 | + signals line | 10 | 59.6 % | 61.6 % |
| 7 | letters A–D, "which part of the paper" framing | – | 10 | 51.5 % | 59.6 % |
| 8 | as 7 | + `Position: chunk i of n` line | 10 | 56.6 % | 59.6 % |
| 9 | as 4 | – | 13 (+ intro, + discussion, + method) | 58.6 % | 56.6 % |
| 10 | as 4, exemplars shown with head **and** tail | – | 10 | 61.6 % | 64.6 % |
| 11 | **as 4 + explicit notes on technical conclusions (→ A) and citation-heavy introductions (→ B)** — *final* | – | 10 | **65.7 %** | 68.7 % |

Observations: (i) the answer label matters — reusing the genre prompt with `Part:` instead of `Genre:` as the answer prefix
collapsed everything onto A (44 %); (ii) the remaining errors are almost all one step off (k=0 conclusions and k=1
introductions predicted as k=2), which costs compute but not recall; (iii) no gold k=2 chunk was ever predicted as k=0,
so the "skip contextualisation" class is not eating body text.

In [7]:
# Corpus-wide prediction for all 382 chunks + what it means for the slice budget compared with fixed k = 3
bias = CALIBRATION_BIAS if USE_CALIBRATION else None
preds = []
t0 = time.perf_counter()
for doc, chunks in CORPUS.items():
    for i, c in enumerate(chunks):
        ev = table_evidence(c["text"])
        if ev:
            k, tier, scores = 3, "table", None
        else:
            scores, _ = slm_scores(c["text"]); k, tier = decide(scores, bias), "slm"
        preds.append({"document": c["document"], "chunk_index": i, "id": c["id"], "k": int(k), "tier": tier,
                      "evidence": ev, "p": (np.round(to_probs(scores), 3).tolist() if scores else None)})
elapsed = time.perf_counter() - t0
json.dump(preds, open(PRED_OUT, "w"), indent=1)
print(f"{len(preds)} chunks in {elapsed:.0f} s ({elapsed/len(preds)*1000:.0f} ms/chunk incl. gate)  ->  {PRED_OUT}")
print("k distribution:", dict(sorted(Counter(p["k"] for p in preds).items())), "| by tier:", dict(Counter(p["tier"] for p in preds)))


def slice_chars(chunks, i, k):
    """Characters of the boundary-extended slice [i-k, i+k] (window kept at 2k+1 chunks where the document allows)."""
    if k == 0:
        return 0                                   # k = 0 means: no contextualisation call at all
    lo, hi = i - k, i + k
    if lo < 0: hi, lo = min(len(chunks) - 1, hi - lo), 0
    if hi > len(chunks) - 1: lo, hi = max(0, lo - (hi - (len(chunks) - 1))), len(chunks) - 1
    return sum(len(c["text"]) for c in chunks[lo:hi + 1])

fixed = dyn = 0
for p in preds:
    ch = CORPUS[p["document"]]
    fixed += slice_chars(ch, p["chunk_index"], 3); dyn += slice_chars(ch, p["chunk_index"], p["k"])
print(f"slice text fed to the contextualiser: fixed k=3 ≈ {fixed/4/1e3:.0f}k tokens, dynamic k ≈ {dyn/4/1e3:.0f}k tokens "
      f"({dyn/fixed:.0%} of fixed; ~4 chars/token)   |   chunks skipped entirely (k=0): {sum(p['k']==0 for p in preds)}")

382 chunks in 60 s (156 ms/chunk incl. gate)  ->  dynamic_slice_length/predicted_k_all_chunks.json
k distribution: {0: 109, 1: 53, 2: 160, 3: 60} | by tier: {'slm': 340, 'table': 42}
slice text fed to the contextualiser: fixed k=3 ≈ 1844k tokens, dynamic k ≈ 972k tokens (53% of fixed; ~4 chars/token)   |   chunks skipped entirely (k=0): 109


## Notes and next steps

* **Where the SLM still fails:** technical conclusions and citation-heavy introductions land in C (k = 2). They are one
  step off and only cost compute. Fragments that read like prose (`even things share massive data, …`) land in B; a
  cheap `starts_mid_sentence` post-rule (first letter lowercase ⇒ at least k = 2) would fix those without a model call.
* **Position is free metadata** (chunk *i* of *n*) but did not help this model as a prompt line; it may work better as a
  post-rule (e.g. last chunks that the SLM calls A are references/bios with high confidence).
* **Calibration** can be re-fitted whenever `slice_radius_gold.json` grows; keep the leave-one-document-out estimate as
  the number to report.
* **Pipeline hook:** `predict_dynamic_k(text)` returns `(k, tier)`; k = 0 means *skip the contextualisation call*, otherwise
  build the slice exactly as in the paper with the predicted radius. `predicted_k_all_chunks.json` already holds the
  radius for every chunk of the benchmark corpus so the retrieval metrics can be recomputed against the same baselines.